# Copy/subcopy redundancy curriculum for the LC edit model

The dataset corruption is intentionally route-level: a redundant route is a
copy or a contiguous subcopy of another route.  We do not inject artificial
`last -> y -> last` loops.  The curriculum starts with obvious whole-route
copies, adds boundary subcopies, then mixed interior subcopies and finally
clean LC seeds so the agent also learns when to halt.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"   # select GPU 1
import sys, pickle, shutil, json, random as _random
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from hydra import compose, initialize_config_dir
from IPython.display import display

from eval_lib.context import (ROOT_DIR, CFG_DIR, DATASETS_DIR,
                              MODEL_OUTPUTS_DIR, EDIT_MODEL_WEIGHTS_DIR)
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))
from connectpt.routes_generator import utils as lrnu
from connectpt.routes_generator.improvement_learning import (
    load_raw_graphs_and_lc_routes, make_improvement_batch,
    rollout_lc_improvement, train_lc_improvement_cfg)
from connectpt.routes_generator.torch_utils import (
    get_batch_tensor_from_routes, dump_routes)
from connectpt.routes_generator.transit_time_estimator import RouteGenBatchState
from connectpt.routes_generator.citygraph_dataset import (
    STOP_KEY, DynamicCityGraphDataset)
from connectpt.routes_generator.bee_colony import get_adjustment_degrees
from torch_geometric.data import Batch
from eval_lib.results_io import save_table
from eval_lib import build_lc_cfg, run_lc, as_route_tensor
from eval_lib import plots as route_plots

pd.set_option("display.max_columns", None)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## Configuration

`copy_full`, `copy_boundary`, `copy_mixed`, and `lc_clean` are equal-size
tiers.  The three corrupted tiers are generated from LC routes with the four
copy/subcopy mutations below.  Multiplicity grows across the curriculum up to
five routes on one stop-to-stop leg.

In [ ]:
# --- dataset: four equal tiers, easy -> hard/clean ---
N_GRAPHS        = 1000
RAW_N_NODES     = 50
RAW_GRAPH_TYPE  = "mixed"
RAW_GRAPH_SEED  = 0
TARGET_N_ROUTES = 12
MIN_ROUTE_LEN   = 8
MAX_ROUTE_LEN   = 15
LC_N_SAMPLES    = 1
LC_COMBOS = [(1.0,0.0,0.0,"demand"), (0.0,1.0,0.0,"route"), (0.0,0.0,1.0,"conn")]

COPY_MUTATION_KINDS = ("full_copy", "prefix_copy", "suffix_copy", "middle_copy")
TIERS = ["copy_full", "copy_boundary", "copy_mixed", "lc_clean"]
TIER_CFG = {
    "copy_full": {
        "events": 3, "max_multiplicity": 3,
        "kinds": ("full_copy",),
    },
    "copy_boundary": {
        "events": 4, "max_multiplicity": 4,
        "kinds": ("prefix_copy", "suffix_copy"),
    },
    "copy_mixed": {
        "events": 6, "max_multiplicity": 5,
        "kinds": COPY_MUTATION_KINDS,
    },
    "lc_clean": {
        "events": 0, "max_multiplicity": 1,
        "kinds": (),
    },
}
DATASET_DIRNAME = "lc_copy_subcopy_curriculum_n50_r12_len8_15_v2"
NEW_DATASET_DIR = DATASETS_DIR / DATASET_DIRNAME
SUBSET_PKL = NEW_DATASET_DIR / "raw_graphs_subset.pkl"
META_CSV   = NEW_DATASET_DIR / "meta.csv"
FORCE_REGEN = False

# --- objective route + connectivity + adjustment cap ---
DISABLED_COST_COMPONENTS = ["demand"]
ADJ_OBJECTIVE = "cap"; ADJ_WEIGHT = 2.0; ADJ_TARGET = 0.15; ADJ_MODE = "paper"; ADJ_GAP = 0.1
VARY_WEIGHTS = True; OP_FRACTION = 0.4; MCW_FRACTION = 0.4

# --- anti-halt-collapse ---
FORCE_NONHALT_FIRST_STEP = True
FORCE_NONHALT_UNTIL_ITER = 8
POSITIVE_ONLY_TRIM_REWARD = True
ENTROPY_WEIGHT = 0.01

# --- critic norm + Huber + value clip ---
CRITIC_OVERRIDES = ["++critic_normalize_returns=true", "++critic_huber=true",
                    "++critic_huber_delta=1.0", "++critic_value_clip=0.2"]

# --- training ---
N_ITERATIONS = 50
BATCH_SIZE   = 16
TRAIN_FRACTION = 0.9
SPLIT_SEED   = 0
MAX_ROUTE_EDIT_STEPS = MAX_ROUTE_LEN
MAX_TRIM_ACTIONS_PER_ROUTE = 1

# --- cumulative curriculum: obvious duplicate -> partial overlap -> mixed -> clean ---
USE_CURRICULUM = True
CURRICULUM = [
    (8,            ["copy_full"],                                      "full-copy"),
    (17,           ["copy_full", "copy_boundary"],                     "+boundary"),
    (33,           ["copy_full", "copy_boundary", "copy_mixed"],       "+mixed"),
    (N_ITERATIONS, TIERS,                                               "+clean"),
]

# --- evaluation ---
EVAL_WEIGHT_COMBOS = [(1.0, 0.0, "route-only"), (0.0, 1.0, "conn-only"), (0.5, 0.5, "balanced")]
EVAL_N_PER_TIER = 10
RUN_NAME = "lc_copy_subcopy_curriculum_route_conn_adj"
print(f"{N_GRAPHS} graphs, tiers={TIERS}; curriculum={[c[2]+'<'+str(c[0]) for c in CURRICULUM]}")
print(f"copy mutations={COPY_MUTATION_KINDS}")
print(f"adj cap t={ADJ_TARGET} W={ADJ_WEIGHT}; force_nonhalt<{FORCE_NONHALT_UNTIL_ITER}; "
      f"positive_only_trim_reward={POSITIVE_ONLY_TRIM_REWARD}; entropy={ENTROPY_WEIGHT}")

## Generate the four-tier route-copy dataset

Every corruption copies a whole donor route or a contiguous donor subroute
into another route slot.  Prefix, suffix, and interior replacements preserve
the recipient length.  Candidates with repeated stops are rejected, so the
dataset teaches inter-route redundancy rather than synthetic self-loops.

In [ ]:
def _to_fixed(routes):
    t = as_route_tensor(routes).long()
    if t.ndim == 3:
        t = t[0]
    if t.shape[0] < TARGET_N_ROUTES:
        t = torch.cat([t, torch.full((TARGET_N_ROUTES - t.shape[0], t.shape[1]), -1, dtype=t.dtype)], 0)
    else:
        t = t[:TARGET_N_ROUTES]
    if t.shape[1] < MAX_ROUTE_LEN:
        t = torch.cat([t, torch.full((t.shape[0], MAX_ROUTE_LEN - t.shape[1]), -1, dtype=t.dtype)], 1)
    elif t.shape[1] > MAX_ROUTE_LEN:
        t = t[:, :MAX_ROUTE_LEN]
    return t


def _tensors(g):
    return {"node_locs": g[STOP_KEY].pos.detach().cpu().clone(),
            "street_adj": g.street_adj.detach().cpu().clone(),
            "demand": g.demand.detach().cpu().clone()}


def _route_nodes(route):
    return [int(node) for node in route.tolist() if int(node) >= 0]


def _leg_counts(routes):
    counts = Counter()
    for route in routes:
        nodes = _route_nodes(route)
        for start, end in zip(nodes[:-1], nodes[1:]):
            counts[(min(start, end), max(start, end))] += 1
    return counts


def _redundancy_stats(routes):
    counts = _leg_counts(routes)
    traversals = sum(counts.values())
    redundancy = 0.0 if traversals == 0 else (traversals - len(counts)) / traversals
    return {
        "redundancy": float(redundancy),
        "max_leg_use": max(counts.values(), default=0),
        "edge_traversals": traversals,
        "unique_edges": len(counts),
    }


def _is_simple_route(nodes):
    return (MIN_ROUTE_LEN <= len(nodes) <= MAX_ROUTE_LEN and
            len(nodes) == len(set(nodes)) and
            all(a != b for a, b in zip(nodes[:-1], nodes[1:])))


def _copy_candidate(routes, donor_idx, target_idx, kind, rng):
    donor = _route_nodes(routes[donor_idx])
    target = _route_nodes(routes[target_idx])
    if not donor or not target:
        return None
    if kind == "full_copy":
        candidate = donor
    else:
        max_seg_len = min(len(donor), len(target), 6)
        if kind == "middle_copy":
            max_seg_len = min(max_seg_len, len(target) - 2)
        if max_seg_len < 2:
            return None
        seg_len = rng.randint(2, max_seg_len)
        if kind == "prefix_copy":
            candidate = donor[:seg_len] + target[seg_len:]
        elif kind == "suffix_copy":
            candidate = target[:-seg_len] + donor[-seg_len:]
        elif kind == "middle_copy":
            donor_start = rng.randint(0, len(donor) - seg_len)
            target_start = rng.randint(1, len(target) - seg_len - 1)
            candidate = (target[:target_start] +
                         donor[donor_start:donor_start + seg_len] +
                         target[target_start + seg_len:])
        else:
            raise ValueError(f"unknown mutation kind: {kind}")
    if candidate == target or not _is_simple_route(candidate):
        return None
    return candidate


def _replace_route(routes, route_idx, nodes):
    routes[route_idx] = -1
    routes[route_idx, :len(nodes)] = torch.as_tensor(nodes, dtype=routes.dtype)


def _try_copy_mutation(routes, kind, max_multiplicity, rng, attempts=80):
    '''Copy one donor route/subroute into 1..max_multiplicity-1 recipients.'''
    base = routes.clone()
    for _ in range(attempts):
        donor_idx = rng.randrange(routes.shape[0])
        target_idxs = [idx for idx in range(routes.shape[0]) if idx != donor_idx]
        rng.shuffle(target_idxs)
        wanted = rng.randint(1, min(max_multiplicity - 1, len(target_idxs)))
        mutated = base.clone()
        current_redundancy = _redundancy_stats(mutated)["redundancy"]
        touched = 0
        for target_idx in target_idxs:
            candidate = _copy_candidate(mutated, donor_idx, target_idx, kind, rng)
            if candidate is None:
                continue
            proposal = mutated.clone()
            _replace_route(proposal, target_idx, candidate)
            proposal_redundancy = _redundancy_stats(proposal)["redundancy"]
            if proposal_redundancy <= current_redundancy + 1e-12:
                continue
            mutated = proposal
            current_redundancy = proposal_redundancy
            touched += 1
            if touched >= wanted:
                return mutated, touched
    return base, 0


def inject_route_copy_redundancy(routes, tier_cfg, rng):
    routes = routes.clone()
    applied_events = Counter()
    mutated_routes = Counter()
    allowed_kinds = tuple(tier_cfg["kinds"])
    for _ in range(int(tier_cfg["events"])):
        candidates = list(allowed_kinds)
        rng.shuffle(candidates)
        for kind in candidates:
            proposal, touched = _try_copy_mutation(
                routes, kind, int(tier_cfg["max_multiplicity"]), rng)
            if touched:
                routes = proposal
                applied_events[kind] += 1
                mutated_routes[kind] += touched
                break
    return routes, applied_events, mutated_routes


def generate_dataset():
    if NEW_DATASET_DIR.exists():
        shutil.rmtree(NEW_DATASET_DIR)
    NEW_DATASET_DIR.mkdir(parents=True, exist_ok=True)
    _random.seed(RAW_GRAPH_SEED); torch.manual_seed(RAW_GRAPH_SEED)
    ds = DynamicCityGraphDataset(min_nodes=RAW_N_NODES, max_nodes=RAW_N_NODES,
                                 data_type=RAW_GRAPH_TYPE, mumford_style=True, pos_only=False)
    raw = [ds.generate_graph(n_nodes=RAW_N_NODES) for _ in range(N_GRAPHS)]
    per = N_GRAPHS // len(TIERS)
    subset, meta = [], []
    for gi, g in enumerate(raw):
        tier = TIERS[min(gi // per, len(TIERS) - 1)]
        tier_cfg = TIER_CFG[tier]
        rng = _random.Random(1000 + gi)
        d, rt, cn, ctag = LC_COMBOS[gi % len(LC_COMBOS)]
        c = build_lc_cfg(run_name=f"copy_cur_{gi}", n_routes=TARGET_N_ROUTES,
                         min_route_len=MIN_ROUTE_LEN, max_route_len=MAX_ROUTE_LEN,
                         demand_time_weight=d, route_time_weight=rt, median_connectivity_weight=cn)
        _, _, _, raw_routes, _ = run_lc(c, tensors=_tensors(g), run_name_prefix="copy_cur_",
                                        n_samples=LC_N_SAMPLES)
        routes = _to_fixed(raw_routes)
        before = _redundancy_stats(routes)
        routes, event_counts, route_counts = inject_route_copy_redundancy(routes, tier_cfg, rng)
        after = _redundancy_stats(routes)
        gdir = NEW_DATASET_DIR / f"graph_{gi:04d}"; gdir.mkdir(parents=True, exist_ok=True)
        dump_routes(f"lc_copy_cur_graph_{gi:04d}_routes", routes, out_dir=gdir)
        subset.append(g)
        meta.append({
            "graph_index": gi, "tier": tier, "lc_combo": ctag,
            "requested_events": int(tier_cfg["events"]),
            "applied_events": int(sum(event_counts.values())),
            "mutated_routes": int(sum(route_counts.values())),
            "mutation_events": json.dumps(dict(event_counts), sort_keys=True),
            "mutation_routes": json.dumps(dict(route_counts), sort_keys=True),
            "redun_before": round(before["redundancy"], 4),
            "redun_after": round(after["redundancy"], 4),
            "max_leg_use_before": before["max_leg_use"],
            "max_leg_use_after": after["max_leg_use"],
        })
        if (gi + 1) % 100 == 0:
            print(f"  {gi+1}/{N_GRAPHS} (tier={tier})")
    with SUBSET_PKL.open("wb") as fh:
        pickle.dump(subset, fh)
    pd.DataFrame(meta).to_csv(META_CSV, index=False)
    print(f"Saved {len(subset)} graphs -> {NEW_DATASET_DIR}")


_have = len(list(NEW_DATASET_DIR.glob("graph_*"))) if NEW_DATASET_DIR.exists() else 0
if SUBSET_PKL.exists() and _have == N_GRAPHS and META_CSV.exists() and not FORCE_REGEN:
    print(f"dataset already exists ({_have}) -> skip")
else:
    if _have and _have != N_GRAPHS:
        print(f"found {_have} graphs, expected {N_GRAPHS} -> regenerate")
    generate_dataset()

## Load, split, and define the cumulative curriculum

In [ ]:
graphs, seed_routes = load_raw_graphs_and_lc_routes(SUBSET_PKL, NEW_DATASET_DIR)
meta_df = pd.read_csv(META_CSV)
N = len(graphs)
print(f"loaded {N} graphs; seed_routes={tuple(seed_routes.shape)}")
print("copy/subcopy corruption summary by tier:")
display(meta_df.groupby("tier")[[
    "requested_events", "applied_events", "mutated_routes",
    "redun_before", "redun_after", "max_leg_use_after",
]].mean().round(3).reindex(TIERS))

_perm = torch.randperm(N, generator=torch.Generator().manual_seed(SPLIT_SEED))
_ntr = int(TRAIN_FRACTION * N)
TRAIN_INDICES = _perm[:_ntr].clone()
VAL_INDICES = _perm[_ntr:].clone()
TIER_OF = dict(zip(meta_df["graph_index"], meta_df["tier"]))

_train_by_tier = {tier: [] for tier in TIERS}
for gi in TRAIN_INDICES.tolist():
    _train_by_tier[TIER_OF[gi]].append(gi)
_train_by_tier = {
    tier: torch.tensor(indices, dtype=torch.long)
    for tier, indices in _train_by_tier.items()
}
print("train graphs per tier:", {tier: len(indices) for tier, indices in _train_by_tier.items()})


def curriculum_fn(iteration):
    '''iteration -> (active train indices, stage label).'''
    for until, tiers, label in CURRICULUM:
        if iteration < until:
            idx = torch.cat([_train_by_tier[tier] for tier in tiers if len(_train_by_tier[tier])])
            return idx, label
    tiers = CURRICULUM[-1][1]
    idx = torch.cat([_train_by_tier[tier] for tier in tiers if len(_train_by_tier[tier])])
    return idx, CURRICULUM[-1][2]


def stage_spans():
    '''[(start_iter, end_iter, label)] for curriculum shading.'''
    spans, prev = [], 0
    for until, _tiers, label in CURRICULUM:
        spans.append((prev + 1, until, label)); prev = until
    return spans

## Model and objective

The trim model receives the redundancy-aware edge channels plus two gated
adjustment features: the fixed cap target and the live `Adj(current, seed)`.
The live value is recomputed after every edit with route-slot ordering
preserved.  The ordinary LC construction model remains unchanged.

In [ ]:
overrides = [
    "model=bestsofar_feb2023_trim",
    "model.route_generator.kwargs.serial_halting=True",
    "++model.route_generator.kwargs.n_adjustment_cond_feats=2",
    f"++run_name={RUN_NAME}", "++experiment.logdir=null",
    f"++adjustment_degree_weight={float(ADJ_WEIGHT)}",
    f"++adjustment_degree_target={float(ADJ_TARGET)}",
    f"++adjustment_degree_objective={ADJ_OBJECTIVE}",
    f"++adjustment_degree_gap={float(ADJ_GAP)}",
    f"++adjustment_degree_mode={ADJ_MODE}",
    "++adjustment_conditioning=true",
    "++adjustment_condition_current=true",
    "++adjustment_condition_weight=false",
    f"++adjustment_target_min={float(ADJ_TARGET)}",
    f"++adjustment_target_max={float(ADJ_TARGET)}",
    f"++entropy_weight={float(ENTROPY_WEIGHT)}",
    f"++force_nonhalt_first_step_until_iter={int(FORCE_NONHALT_UNTIL_ITER)}",
    f"++positive_only_trim_reward={str(POSITIVE_ONLY_TRIM_REWARD).lower()}",
] + CRITIC_OVERRIDES
with initialize_config_dir(config_dir=str(CFG_DIR), version_base=None):
    cfg = compose(config_name="ppo_50nodes.yaml", overrides=overrides)
device, run_name, _, cost_obj, model = lrnu.process_standard_experiment_cfg(
    cfg, run_name_prefix="improvement_")
cost_obj.ignore_stops_oob = True
cost_obj.set_enabled_components(disabled_components=DISABLED_COST_COMPONENTS or None)
if VARY_WEIGHTS:
    cost_obj.variable_weights = True
    cost_obj.pp_fraction = 0.0; cost_obj.op_fraction = OP_FRACTION; cost_obj.mcw_fraction = MCW_FRACTION
BEST_MODEL_PATH = EDIT_MODEL_WEIGHTS_DIR / f"{run_name}.pt"
print(f"run_name={run_name} | enabled={list(cost_obj.enabled_component_names)} | "
      f"variable_weights={cost_obj.variable_weights}")
print(f"redundancy edge_dim={model.edge_feat_dim}; adj conditioning=[target, current_adj]")
print(f"critic norm={cfg.get('critic_normalize_returns')} huber={cfg.get('critic_huber')} clip={cfg.get('critic_value_clip')}")

## Train with cumulative copy/subcopy curriculum

In [ ]:
train_result = train_lc_improvement_cfg(
    model=model, cost_obj=cost_obj, graphs=graphs, seed_routes=seed_routes,
    device=device, cfg=cfg, output_dir=MODEL_OUTPUTS_DIR, run_name=run_name,
    train_fraction=TRAIN_FRACTION, batch_size=BATCH_SIZE,
    min_route_len=MIN_ROUTE_LEN, max_route_len=MAX_ROUTE_LEN, seed=SPLIT_SEED,
    max_route_edit_steps=MAX_ROUTE_EDIT_STEPS,
    max_trim_actions_per_route=MAX_TRIM_ACTIONS_PER_ROUTE,
    target_n_routes=TARGET_N_ROUTES,
    train_indices=TRAIN_INDICES, val_indices=VAL_INDICES,
    best_model_path=BEST_MODEL_PATH, n_iterations=N_ITERATIONS,
    force_nonhalt_first_step=FORCE_NONHALT_FIRST_STEP,
    curriculum_fn=(curriculum_fn if USE_CURRICULUM else None),
)
history_df = pd.DataFrame(train_result["history"])
save_table(history_df, f"{RUN_NAME}_training_history")
print(f"history rows={len(history_df)}; best -> {BEST_MODEL_PATH}")

## Кривые актора (с границами стадий curriculum)

In [ ]:
h = history_df
def _num(col):
    return pd.to_numeric(h[col], errors="coerce") if col in h.columns else None

_spans = stage_spans()
_colors = ["#eaf3ff", "#eafbea", "#fff6e6", "#fdeaea", "#f0eaff"]
def _shade(ax):
    for k, (s, e, lab) in enumerate(_spans):
        ax.axvspan(s, e, color=_colors[k % len(_colors)], alpha=0.6, zorder=0)
        ax.axvline(s, color="gray", lw=0.6, ls=":")

fig, ax = plt.subplots(2, 3, figsize=(17, 8), constrained_layout=True)
panels = [("train_reward_mean","train reward"), ("val_delta","val cost delta (+=улучш.)"),
          ("val_win_rate","val win rate"), ("train_action_avg_actions_per_route","avg edits/route"),
          ("val_component_delta_route","val route delta"),
          ("val_component_delta_connectivity","val conn delta")]
for a,(col,title) in zip(ax.flat, panels):
    _shade(a); y=_num(col)
    if y is not None and y.notna().any():
        a.plot(h["epoch"], y, marker="o", ms=2, color="tab:blue", zorder=3)
    a.axhline(0,color="k",lw=0.7); a.set_title(title); a.set_xlabel("epoch"); a.grid(alpha=0.2)
# подписи стадий сверху
for s,e,lab in _spans:
    ax[0,0].text((s+e)/2, ax[0,0].get_ylim()[1], lab, ha="center", va="bottom", fontsize=8)
fig.suptitle("Actor curves + curriculum stages (заливка = стадия)", fontsize=13, fontweight="bold")
plt.show(); plt.close(fig)

## Метрики критика (с границами стадий)

In [ ]:
crit_cols = [c for c in h.columns if "critic" in c.lower()]
print("critic columns:", crit_cols)
if crit_cols:
    n=len(crit_cols)
    fig, ax = plt.subplots(1, n, figsize=(5*n, 4), squeeze=False, constrained_layout=True)
    for a, col in zip(ax[0], crit_cols):
        _shade(a); y=_num(col)
        if y is not None and y.notna().any():
            a.plot(h["epoch"], y, marker="o", ms=2, color="tab:orange", zorder=3)
        a.set_title(col, fontsize=9); a.set_xlabel("epoch"); a.grid(alpha=0.2)
        if "explained" in col: a.axhline(0, color="k", lw=0.7)
    fig.suptitle("Critic metrics + curriculum stages", fontsize=13, fontweight="bold")
    plt.show(); plt.close(fig)
    display(h[["epoch","curriculum_stage"]+crit_cols].iloc[::max(1,len(h)//15)].round(4))

## Evaluate by tier and preference weights

The table reports route-level redundancy, `ATT`, `RTT`, connectivity, and
`Adj(current, seed)`.  Positive `*_drop` values mean improvement.

In [ ]:
def _redun_t(routes_2d):
    return _redundancy_stats(routes_2d)["redundancy"]


def _mean_metric(result, key):
    value = result.get_metrics()[key]
    return float(value.detach().float().mean().item())


val_by_tier = {tier: [] for tier in TIERS}
for gi in VAL_INDICES.tolist():
    tier = TIER_OF[gi]
    if len(val_by_tier[tier]) < EVAL_N_PER_TIER:
        val_by_tier[tier].append(gi)

base_w = cost_obj.get_weights(device)
def mkw(route_weight, conn_weight):
    weights = {key: (value.clone() if torch.is_tensor(value) else value)
               for key, value in base_w.items()}
    weights["demand_time_weight"] = torch.as_tensor(0.0, device=device)
    weights["route_time_weight"] = torch.as_tensor(float(route_weight), device=device)
    weights["median_connectivity_weight"] = torch.as_tensor(float(conn_weight), device=device)
    return weights


rows = []
visual_examples = {}
conn_metric_key = ("median_connectivity_weighted"
                   if cost_obj.use_weighted_connectivity
                   else "median_connectivity")
model.eval()
for route_weight, conn_weight, weight_tag in EVAL_WEIGHT_COMBOS:
    weights = mkw(route_weight, conn_weight)
    for tier in TIERS:
        idxs = val_by_tier[tier]
        if not idxs:
            continue
        metrics = {
            "redun_seed": [], "redun_after": [], "Adj": [],
            "ATT_seed": [], "ATT_after": [],
            "RTT_seed": [], "RTT_after": [],
            "CONN_seed": [], "CONN_after": [],
        }
        for gi in idxs:
            graph_batch, route_batch = make_improvement_batch(
                graphs, seed_routes, torch.tensor([gi]), device,
                training=False, target_n_routes=TARGET_N_ROUTES)
            with torch.no_grad():
                output = rollout_lc_improvement(
                    model, cost_obj, graph_batch, route_batch,
                    MIN_ROUTE_LEN, MAX_ROUTE_LEN,
                    greedy=True, cost_weights=weights,
                    max_route_edit_steps=MAX_ROUTE_EDIT_STEPS,
                    max_trim_actions_per_route=MAX_TRIM_ACTIONS_PER_ROUTE,
                    adjustment_target=ADJ_TARGET,
                    adjustment_use_current=True,
                    adjustment_gap=ADJ_GAP,
                    adjustment_mode=ADJ_MODE)
            final_state, seed_result, final_result = output[:3]
            improved = get_batch_tensor_from_routes(
                final_state.routes, device, max_route_len=route_batch.shape[-1])
            nr = min(improved.shape[1], route_batch.shape[1])
            width = min(improved.shape[-1], route_batch.shape[-1])
            adj = get_adjustment_degrees(
                improved[:, :nr, :width], route_batch[:, :nr, :width],
                cost_obj.symmetric_routes, gap=ADJ_GAP, mode=ADJ_MODE
            ).mean().item()

            metrics["redun_seed"].append(_redun_t(route_batch[0]))
            metrics["redun_after"].append(_redun_t(improved[0]))
            metrics["Adj"].append(adj)
            metrics["ATT_seed"].append(_mean_metric(seed_result, "ATT"))
            metrics["ATT_after"].append(_mean_metric(final_result, "ATT"))
            metrics["RTT_seed"].append(_mean_metric(seed_result, "RTT"))
            metrics["RTT_after"].append(_mean_metric(final_result, "RTT"))
            metrics["CONN_seed"].append(_mean_metric(seed_result, conn_metric_key))
            metrics["CONN_after"].append(_mean_metric(final_result, conn_metric_key))

            if weight_tag == "balanced" and tier not in visual_examples:
                visual_examples[tier] = {
                    "graph_index": gi,
                    "seed": route_batch[0].detach().cpu(),
                    "improved": improved[0].detach().cpu(),
                    "Adj": adj,
                    "redun_seed": metrics["redun_seed"][-1],
                    "redun_after": metrics["redun_after"][-1],
                    "ATT_seed": metrics["ATT_seed"][-1],
                    "ATT_after": metrics["ATT_after"][-1],
                    "RTT_seed": metrics["RTT_seed"][-1],
                    "RTT_after": metrics["RTT_after"][-1],
                    "CONN_seed": metrics["CONN_seed"][-1],
                    "CONN_after": metrics["CONN_after"][-1],
                }

        means = {key: float(np.mean(values)) for key, values in metrics.items()}
        rows.append({
            "eval_weights": weight_tag, "tier": tier, "n": len(idxs),
            "redun_seed": means["redun_seed"],
            "redun_after": means["redun_after"],
            "redun_drop": means["redun_seed"] - means["redun_after"],
            "ATT_seed": means["ATT_seed"], "ATT_after": means["ATT_after"],
            "ATT_drop": means["ATT_seed"] - means["ATT_after"],
            "RTT_seed": means["RTT_seed"], "RTT_after": means["RTT_after"],
            "RTT_drop": means["RTT_seed"] - means["RTT_after"],
            "CONN_seed": means["CONN_seed"], "CONN_after": means["CONN_after"],
            "CONN_drop": means["CONN_seed"] - means["CONN_after"],
            "Adj": means["Adj"],
        })

eval_df = pd.DataFrame(rows).round(4)
display(eval_df)
save_table(eval_df, f"{RUN_NAME}_eval_by_tier")
print("redun_drop>0 means that copied route coverage was removed.")
print("ATT/RTT/CONN are reported in minutes; Adj is mean route-wise change vs seed.")

## Visual validation examples

For the balanced preference vector, show one seed and the corresponding edited
network from every tier.  The right-hand panels emphasize removed and added
segments relative to the corrupted seed.

In [ ]:
if not visual_examples:
    print("Run the evaluation cell first.")
else:
    tiers_to_plot = [tier for tier in TIERS if tier in visual_examples]
    fig, axes = plt.subplots(
        len(tiers_to_plot), 2,
        figsize=(18, 7 * len(tiers_to_plot)),
        squeeze=False, constrained_layout=True)
    for row_idx, tier in enumerate(tiers_to_plot):
        example = visual_examples[tier]
        graph = graphs[example["graph_index"]]
        route_plots.plot_plain_route_set(
            axes[row_idx, 0], example["seed"], graph,
            title=f"{tier}: corrupted seed (graph {example['graph_index']})",
            subtitle=(f"redun={example['redun_seed']:.3f}; "
                      f"ATT={example['ATT_seed']:.2f}; RTT={example['RTT_seed']:.2f}; "
                      f"CONN={example['CONN_seed']:.2f}"))
        route_plots.plot_route_diff(
            axes[row_idx, 1], example["improved"], example["seed"], graph,
            title=f"{tier}: edited network vs seed",
            subtitle=(f"redun {example['redun_seed']:.3f}->{example['redun_after']:.3f}; "
                      f"Adj={example['Adj']:.3f}\n"
                      f"ATT {example['ATT_seed']:.2f}->{example['ATT_after']:.2f}; "
                      f"RTT {example['RTT_seed']:.2f}->{example['RTT_after']:.2f}; "
                      f"CONN {example['CONN_seed']:.2f}->{example['CONN_after']:.2f}"))
    fig.suptitle("Balanced validation: copy/subcopy corruption repair", fontsize=15, fontweight="bold")
    plt.show()
    plt.close(fig)